<a href="https://colab.research.google.com/github/SOMA-AlsheiKH/cosc726-SomiaMohammedSaidahmed/blob/main/week02_Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COSC726 · Lab 1 — LLM Foundations as Measurable Systems

**Agentic Artificial Intelligence · Week 2 · 2-hour guided lab**

Last week you read an agent from the outside. This week you open the **reasoning component** — the LLM —
and treat it as an object of *measurement*, not a magic box. Everything here runs **offline, on the Python
standard library, with no API key**: the model is a deterministic, transparent stand-in so the *mechanisms*
are visible and reproducible on every machine.

Running example throughout: the customer-support agent helping **Layla** with order **#A1032**.

**Six investigations:** tokenization · context budgets · sampling · state · grounding · telemetry.
**Submit:** this notebook executed top-to-bottom, your passing `--self-test`, and six observations.

## 0 · The layer we are studying

A model call maps *constructed context* to *generated output*. It does **not** own state, tools, permissions,
validation, or termination — the application does. Keep this boundary in mind: most bugs you meet this term
live in the harness, not the model.

In [ ]:
# Two DELIBERATELY DIFFERENT teaching tokenizers. Neither is authoritative — that is the point.
def greedy_split(text, vocab):
    vocab = sorted(vocab, key=len, reverse=True)
    text, out, i = text.lower(), [], 0
    while i < len(text):
        for v in vocab:
            if v and text.startswith(v.lower(), i):
                out.append(v); i += len(v); break
        else:
            out.append(text[i]); i += 1
    return out

TOK_A = ["order","agent","the","credit","policy","late","1043","ic","ing","ed"," ","-","A","#"]
TOK_B = ["order","ag","ent","the","cred","it","pol","icy","late","10","43","ic","ing","ed"," ","-","A","#"]

for label, vocab in [("A", TOK_A), ("B", TOK_B)]:
    toks = greedy_split("order A-1043", vocab)
    print(f"tokenizer {label}: {len(toks)} tokens  {toks}")

tokenizer A: 5 tokens  ['order', ' ', 'A', '-', '1043']
tokenizer B: 6 tokens  ['order', ' ', 'A', '-', '10', '43']


## 1 · Tokenization is model-specific

The same text, `"order A-1043"`, costs a **different number of tokens** under each tokenizer. There is no
universal "one token ≈ four characters" rule. Below, compare English, Arabic, an identifier, and a JSON
payload — the kinds of input Layla's agent really sees.

In [ ]:
samples = {
    "English":    "the late credit policy",
    "Arabic":     "الذكاء الاصطناعي القائم على الوكلاء",   # counted as raw characters by our toy tokenizer
    "Identifier": "customer_order_A-1043",
    "JSON":       '{"status":"pending_approval"}',
}
print(f"{'input':<12}{'tok A':>7}{'tok B':>7}   provenance you MUST record")
print("-" * 60)
for name, text in samples.items():
    a = len(greedy_split(text, TOK_A))
    b = len(greedy_split(text, TOK_B))
    print(f"{name:<12}{a:>7}{b:>7}   (tokenizer + version + the exact text)")
print("\nObservation 1: which input is most expensive, and why? A count WITHOUT")
print("its tokenizer/version is not reproducible evidence.")

input         tok A  tok B   provenance you MUST record
------------------------------------------------------------
English           7      9   (tokenizer + version + the exact text)
Arabic           35     35   (tokenizer + version + the exact text)
Identifier       14     15   (tokenizer + version + the exact text)
JSON             27     27   (tokenizer + version + the exact text)

Observation 1: which input is most expensive, and why? A count WITHOUT
its tokenizer/version is not reproducible evidence.


### **Observation 1: Tokenization Analysis**

* **Most Expensive Input:**  
  The **Arabic** input (`"الذكاء الاصطناعي القائم على الوكلاء"`), which consumed **35 tokens** under both tokenizers (`tok A` and `tok B`)[cite: 2].

* **Why:**  
  Because the vocabulary lists defined for both teaching tokenizers (`TOK_A` and `TOK_B`) do not contain any Arabic words or characters[cite: 2]. Consequently, the greedy split algorithm failed to find vocabulary matches and fell back to counting **character-by-character** (raw character fallback)[cite: 2]. This caused every individual character to be treated as a separate token, drastically inflating the token count compared to English[cite: 2].

* **Why Provenance & Versioning Matter:**  
  A "token" is not a universal or standardized unit of measurement[cite: 1, 2]. Token counts depend entirely on the specific **tokenizer**, its **vocabulary list**, and its **version**[cite: 1, 2]. Reporting a token count without recording the exact tokenizer name, version, and text input renders the measurement unverified and impossible to reproduce or audit scientifically[cite: 1, 2].

## 2 · The context window is a budget — overflow is a policy

The window holds instructions + messages + retrieval + tools + output + reasoning, all at once. When the
input would exceed the budget, real APIs commonly **reject** the request — they do not silently trim. Whether
you drop, summarise, or retrieve is a **policy you choose and log**.

In [ ]:
def count_tokens(text, vocab):
    return len(greedy_split(text, vocab))

def prepare_context(messages, context_limit, reserved_output, vocab, strategy="drop_oldest"):
    budget = context_limit - reserved_output
    total = lambda ms: sum(count_tokens(m, vocab) for m in ms)
    if strategy == "reject":
        if total(messages) > budget:
            return {"rejected": True, "reason": f"input {total(messages)} > budget {budget}",
                    "kept": [], "dropped": []}
        return {"rejected": False, "reason": "", "kept": list(messages), "dropped": []}
    kept, dropped = list(messages), []
    while total(kept) > budget and len(kept) > 1:
        dropped.append(kept.pop(1))            # keep messages[0] (system) always
    return {"rejected": total(kept) > budget, "reason": "", "kept": kept, "dropped": dropped}

convo = ["SYSTEM: you are Layla's support agent",
         "USER: where is order #A1032?",
         "ASSISTANT: checking now",
         "USER: it is very late, do I get a credit?"]

print("REJECT policy (tight budget):")
print(" ", prepare_context(convo, context_limit=12, reserved_output=6, vocab=TOK_A, strategy="reject"))
print("\nDROP-OLDEST policy (same messages, logged):")
plan = prepare_context(convo, context_limit=40, reserved_output=6, vocab=TOK_A, strategy="drop_oldest")
print("  kept:   ", plan["kept"])
print("  dropped:", plan["dropped"], " <- LOGGED, never silent")
print("\nObservation 2: for Layla, which message would be UNSAFE to drop silently?")

REJECT policy (tight budget):
  {'rejected': True, 'reason': 'input 111 > budget 6', 'kept': [], 'dropped': []}

DROP-OLDEST policy (same messages, logged):
  kept:    ["SYSTEM: you are Layla's support agent"]
  dropped: ['USER: where is order #A1032?', 'ASSISTANT: checking now', 'USER: it is very late, do I get a credit?']  <- LOGGED, never silent

Observation 2: for Layla, which message would be UNSAFE to drop silently?


### **Observation 2: Context Window & Overflow Policy Analysis**

* **Unsafe Message to Drop:**  
  The message `USER: where is order #A1032?`[cite: 2].

* **Why it is Unsafe:**  
  This message contains the essential anchor entity for the entire interaction: **Order ID (`#A1032`)**[cite: 2]. If this message is dropped silently, the model loses the order reference entirely while trying to process follow-up requests like "do I get a credit?"[cite: 2]. The agent would no longer know which order Layla is referring to, making any financial or credit action unsafe, inaccurate, or prone to hallucination[cite: 2].

* **Key Takeaway:**  
  Context window truncation must never happen silently[cite: 1, 2]. Dropped items must be logged explicitly so the application layer can audit context loss and prevent actions when crucial context identifiers are removed[cite: 1, 2].

## 3 · Sampling: temperature changes diversity, not truth

A model outputs a distribution over the next token; a **sampler** picks one. Temperature reshapes the
distribution before picking. At temperature 0 our sampler is **greedy** — the same token wins every time.
That is *repeatability of the sampler*, not reproducibility of the whole system (more in §5).

In [ ]:
import math, random

def sample_next(distribution, temperature, rng):
    if temperature <= 0:
        top = max(distribution.values())
        return sorted(k for k, v in distribution.items() if v == top)[0]
    scaled = {k: math.exp(math.log(v) / temperature) for k, v in distribution.items() if v > 0}
    z = sum(scaled.values())
    r, acc = rng.random(), 0.0
    for k in sorted(scaled):
        acc += scaled[k] / z
        if r <= acc:
            return k
    return sorted(scaled)[-1]

dist = {"Paris": 0.82, "London": 0.11, "Lyon": 0.05, "Rome": 0.02}
print("temperature 0 (greedy) over 5 seeds:",
      {sample_next(dist, 0.0, random.Random(s)) for s in range(5)}, " <- always the mode")
print("temperature 1.0 over 8 seeds:      ",
      [sample_next(dist, 1.0, random.Random(s)) for s in range(8)])
print("\nObservation 3: for Layla's ORDER-ID extraction, is high or low temperature safer — and how")
print("would you JUSTIFY the choice with evidence rather than taste?")

temperature 0 (greedy) over 5 seeds: {'Paris'}  <- always the mode
temperature 1.0 over 8 seeds:       ['Paris', 'Lyon', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris']

Observation 3: for Layla's ORDER-ID extraction, is high or low temperature safer — and how
would you JUSTIFY the choice with evidence rather than taste?


### **Observation 3: Sampling & Temperature Analysis**

* **Safer Choice for Order-ID Extraction:**  
  **Low temperature (0.0 / Greedy sampling)** is significantly safer[cite: 1, 2].

* **Why Low Temperature is Safer:**  
  Extracting structured identifiers (like Order ID `#A1032`) is a factual precision task that requires deterministic, repeatable outputs rather than creative variation[cite: 1, 2]. At temperature 0, the sampler greedily selects the mode (the highest probability token) every time, eliminating random sampling variance[cite: 1, 2].

* **Justifying with Evidence rather than Taste:**  
  To justify this choice empirically rather than by taste, one would measure:
  1. **Determinism/Variance Rate:** Testing extraction across multiple seeds ($N$ runs)[cite: 1, 2]. At temperature 0, output variance across seeds is $0\%$, whereas higher temperatures introduce stochastic variation (as seen in the `temperature 1.0` run returning different tokens)[cite: 2].
  2. **Exact-Match Accuracy:** Evaluating extracted IDs against ground-truth labels across a test dataset[cite: 1]. High temperature flattens the probability distribution, increasing the error rate by occasionally selecting lower-probability, incorrect tokens for the ID string[cite: 1, 2].

## 4 · State lives in the application, not the model

"The model remembers" is shorthand. A bare call is **stateless**; some component (client or provider)
stores the transcript and decides what becomes context. Watch a bare call forget, then a manager remember.

In [ ]:
def bare_call(context_messages):
    """Stateless: sees ONLY what it is handed, nothing else."""
    for m in context_messages:
        if "layla" in m.lower():
            return "I can see this is Layla."
    return "I do not know who you are."   # not broken — it was simply handed no history

class ConversationManager:
    """The application component that actually stores and reconstructs state."""
    def __init__(self): self.history = []
    def add(self, msg): self.history.append(msg)
    def call(self): return bare_call(self.history)

print("Bare call, no history handed in:  ", bare_call(["what is my name?"]))
mgr = ConversationManager()
mgr.add("Hi, I'm Layla, about order #A1032")
mgr.add("what is my name?")
print("Managed call, history reconstructed:", mgr.call())
print("\nObservation 4: name the component responsible for retention, deletion, and privacy here.")

Bare call, no history handed in:   I do not know who you are.
Managed call, history reconstructed: I can see this is Layla.

Observation 4: name the component responsible for retention, deletion, and privacy here.


### **Observation 4: State Management & Application Responsibility**

* **Responsible Component:**  
  The **`ConversationManager`** (Application Layer / Harness)[cite: 2].

* **Role & Justification:**  
  * **Retention:** The model call (`bare_call`) is purely **stateless** and retains no context or memory between executions[cite: 2]. The `ConversationManager` is responsible for storing, maintaining, and reconstructing history (`self.history`) for subsequent model calls[cite: 2].
  * **Deletion & Privacy:** Because state lives within the application rather than the LLM, the `ConversationManager` is the sole component empowered to delete session history, apply retention limits, scrub sensitive customer identifiers (PII/privacy compliance), and control what context is exposed to the model[cite: 1, 2].

## 5 · Grounding: unsupported generation is an evidence problem

A fluent claim is not a supported one. When an agent may **act** (apply Layla's credit!), an unsupported
fact or an unconfirmed action is unsafe. The fix is a contract: require evidence, or **abstain**.

In [ ]:
POLICY_DB = {"late_delivery": {"text": "Orders >3 days late qualify for a 10% credit.",
                               "evidence_id": "policy-late-delivery-v3"}}

def answer_with_grounding(question, require_evidence=True):
    hit = POLICY_DB.get("late_delivery") if "credit" in question or "late" in question else None
    if hit:
        return {"answer": hit["text"], "evidence_id": hit["evidence_id"], "abstained": False}
    if require_evidence:
        return {"answer": "Insufficient evidence — escalating to a human.",
                "evidence_id": None, "abstained": True}
    return {"answer": "Sure, Layla probably gets some money back!", "evidence_id": None, "abstained": False}

print("grounded  :", answer_with_grounding("is Layla's order eligible for a late credit?"))
print("no evidence, abstain :", answer_with_grounding("what is the CEO's home address?", require_evidence=True))
print("no evidence, ungrounded (UNSAFE):", answer_with_grounding("what is the CEO's home address?", require_evidence=False))
print("\nObservation 5: which of these three could safely drive an ACTION on Layla's account?")

grounded  : {'answer': 'Orders >3 days late qualify for a 10% credit.', 'evidence_id': 'policy-late-delivery-v3', 'abstained': False}
no evidence, abstain : {'answer': 'Insufficient evidence — escalating to a human.', 'evidence_id': None, 'abstained': True}
no evidence, ungrounded (UNSAFE): {'answer': 'Sure, Layla probably gets some money back!', 'evidence_id': None, 'abstained': False}

Observation 5: which of these three could safely drive an ACTION on Layla's account?


### **Observation 5: Grounding, Abstention, & Action Contracts**

* **Safest Response for Driving Action:**  
  Only the **`grounded`** response can safely trigger an automated action (such as issuing a credit) on Layla's account.

* **Justification:**  
  * **Verifiable Audit Trail:** The `grounded` response provides explicit evidence (`evidence_id: "policy-late-delivery-v3"`), creating a verifiable contract between the model's claim and company policy prior to executing state-changing account updates.
  * **Risk of Ungrounded Generation:** The `ungrounded` response lacks proof (`evidence_id: None`). Executing automated account actions on unbacked outputs creates severe financial liabilities and hallucinated policy promises.
  * **Safety via Abstention:** The `abstain` response demonstrates the fallback mechanism—when required evidence is missing, the application harness halts automation and safely escalates to a human operator (`abstained: True`).

## 6 · Telemetry: inspect every model call

A useful trace records enough to estimate cost, reproduce conditions, explain failures, and audit evidence.
This is the normalized response object the course `ModelClient` will standardise (built fully in **Week 4**).

In [ ]:
def mock_generate(question):
    """A transparent, deterministic stand-in that returns a NORMALIZED response object."""
    return {
        "provider_model": "course-mock / mock-llm-v2",
        "request_id": "mock-7f31c2",
        "text": "Order #A1032 is 3 days late; a 10% credit applies (see policy-late-delivery-v3).",
        "usage": {"input_tokens": 118, "output_tokens": 36, "cached": 0, "reasoning": 12, "tool": 8},
        "finish_reason": "completed",
        "context_policy": "drop_oldest",
        "dropped_items": ["message-2"],
        "evidence_ids": ["policy-late-delivery-v3"],
        "abstained": False,
        "latency_ms": 640, "retries": 0,
    }

resp = mock_generate("is Layla's order eligible?")
for k, v in resp.items():
    print(f"  {k:<16} {v}")
print("\nObservation 6: which THREE fields would you need to (a) estimate cost, (b) reproduce")
print("the call, and (c) audit the evidence behind the answer?")

  provider_model   course-mock / mock-llm-v2
  request_id       mock-7f31c2
  text             Order #A1032 is 3 days late; a 10% credit applies (see policy-late-delivery-v3).
  usage            {'input_tokens': 118, 'output_tokens': 36, 'cached': 0, 'reasoning': 12, 'tool': 8}
  finish_reason    completed
  context_policy   drop_oldest
  dropped_items    ['message-2']
  evidence_ids     ['policy-late-delivery-v3']
  abstained        False
  latency_ms       640
  retries          0

Observation 6: which THREE fields would you need to (a) estimate cost, (b) reproduce
the call, and (c) audit the evidence behind the answer?


### **Observation 6: Telemetry & Tracing Analysis**

* **Three Required Fields:**
  1. **(a) To Estimate Cost:** `usage`  
     *(Contains token usage metrics—specifically `input_tokens` and `output_tokens`—required to calculate API billing against provider rate cards).*
  2. **(b) To Reproduce the Call:** `provider_model`  
     *(Identifies the specific provider and exact model version used to process the invocation).*
  3. **(c) To Audit the Evidence:** `evidence_ids`  
     *(Contains the list of explicit policy keys or document IDs that grounded the generated output).*

* **Key Takeaway:**  
  Normalized telemetry metadata enables complete system observability, allowing developers to track operational expenses, debug system behavior across model versions, and ensure policy compliance through audit trails.

## 7 · Wire it up — pass the self-test

Open `COSC726_W02_llm_foundations.py`, implement the three TODOs (`count_tokens`, `prepare_context`,
`sample_next`), then run from a terminal:

```bash
python COSC726_W02_llm_foundations.py --self-test    # target: ALL SELF-TESTS PASSED
```

### Submission checklist
- [ ] This notebook runs top-to-bottom (`Kernel → Restart & Run All`)
- [ ] Your student script prints **ALL SELF-TESTS PASSED**
- [ ] Your **six observations** are written up (one per investigation above)
- [ ] Pushed to your classroom repo under `week02/`

### 🔧 Stretch (optional)
Route **one** approved local or hosted endpoint through a tiny adapter that returns the *same normalized
object* as `mock_generate` — synthetic data only, no real customer text. This previews the provider-adapter
pattern; the full `ModelClient` seam is built in **Lab 3 (Week 4)**.

---
*Next week: prompt & context engineering as behaviour specification — the same model, specified deliberately.*